# Highway and Residual Networks

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

本文中，我们将探索能够训练更深层网络的架构技术。标准的卷积网络随着层数增加会出现梯度消失/爆炸问题，导致训练困难。Highway Networks 和 Residual Networks 通过引入跳跃连接（skip connections）来解决这个问题，使得梯度能够直接传播。

In this notebook we'll explore architectural techniques that enable training of much deeper networks. Standard convolutional networks suffer from vanishing/exploding gradients as the number of layers increase which makes training difficult. Highway Networks and Residual Networks address this by introducing skip connections that allow gradients to flow directly through the network.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/resnet.png" width=650>

# 概述 Overview

* **目标:** 训练更深层的神经网络，同时解决梯度消失/爆炸问题。
* **优点:** 
  * 允许梯度直接传播
  * 可以训练数百甚至上千层的网络
  * 性能显著优于标准网络
* **缺点:**
  * 参数量增加
  * 计算成本增加
* **其他:** 
  * ResNet 是 ImageNet 比赛的冠军架构
  * 启发了许多后续架构如 DenseNet、FractalNet 等

# 设置 Setup

In [ ]:
# Load PyTorch library
!pip3 install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

# 残差块 Residual Block

残差块是 ResNet 的核心组件。输入 x 通过两条路径：
1. 主体路径：进行多层非线性变换
2. 捷径路径：直接传递输入 x

输出为 $y = F(x) + x$，其中 $F$ 是需要学习的映射。

A residual block is the core component of ResNet. The input x goes through two paths:
1. Main path: multi-layer nonlinear transformations
2. Skip path: directly passes input x

The output is $y = F(x) + x$, where $F$ is the mapping to learn.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        
        # 主路径 Main path
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 如果输入输出尺寸不同，需要调整 shortcut
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # 主路径 Main path
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # 捷径连接 Skip connection
        out += self.shortcut(x)
        out = F.relu(out)
        return out

# ResNet 模型 ResNet Model

In [ ]:
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        
        # 初始卷积 Initial convolution
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # 残差层 Residual layers
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        
        # 全连接层 FC layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# 训练 Training

In [ ]:
# 超参数 Hyperparameters
batch_size = 64
learning_rate = 0.001
num_epochs = 5


In [ ]:
# 数据加载 Data loading
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 初始化模型 Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
# 训练函数 Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(train_loader), 100. * correct / total

def evaluate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(test_loader), 100. * correct / total

In [ ]:
# 训练循环 Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')

# Highway Networks

Highway Networks 早于 ResNet 提出了门控机制来控制信息流。核心公式是：

$y = H(x, W_H) \cdot T(x, W_T) + x \cdot (1 - T(x, W_T))$

其中 $T$ 是变换门控，$(1-T)$ 是携带门控。

Highway Networks preceded ResNet and introduced gating mechanisms to control information flow. The core formula is:

$y = H(x, W_H) \cdot T(x, W_T) + x \cdot (1 - T(x, W_T))$

where $T$ is the transform gate and $(1-T)$ is the carry gate.

In [ ]:
class HighwayBlock(nn.Module):
    def __init__(self, size, num_layers):
        super(HighwayBlock, self).__init__()
        self.num_layers = num_layers
        self.gates = nn.ModuleList()
        
        for i in range(num_layers):
            # 变换门控 Transform gate
            self.gates.append(nn.Linear(size, size))
    
    def forward(self, x):
        for i in range(self.num_layers):
            # 变换路径 Transform path
            transform = F.relu(self.gates[i](x))
            # 携带门控 Carry gate (1 - T)
            carry = torch.sigmoid(self.gates[i](x))
            # 门控组合 Gating combination
            x = transform * carry + x * (1 - carry)
        return x

# TODO

- DenseNet 架构
- 预激活残差块 Pre-activation residual blocks
- 残差网络的变体 Variants of residual networks